# Weather MLOps — Exploration sur données complètes

Ce notebook travaille uniquement sur les lignes **sans aucune valeur manquante** :  
toutes les colonnes Open-Meteo enrichies + features de base + targets sont renseignées.

Résultat : **33 524 lignes × 46 colonnes** — dataset ML-ready.

**Sections :**
1. Chargement & rapport de qualité (gates de validation)
2. Profil du dataset complet
3. Features météo — distributions et relations
4. Features enrichies Open-Meteo
5. Targets — distributions et équilibre des classes
6. Corrélations — features vs targets
7. Patterns saisonniers et géographiques
8. Matrice de corrélation complète
9. Synthèse qualité

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid', palette='tab10')

CSV_PATH = Path('..') / 'data' / 'output' / 'weather_final.csv'
assert CSV_PATH.exists(), f"Fichier introuvable : {CSV_PATH}"

df_raw = pd.read_csv(CSV_PATH, parse_dates=['date'])

COLS_ENRICHED = [
    'rain_sum', 'precipitation_hours',
    'dew_point_9am', 'dew_point_3pm',
    'surface_pressure_9am', 'surface_pressure_3pm',
    'shortwave_radiation_sum',
    'vpd_9am', 'vpd_3pm',
    'wind_speed_100m_9am', 'wind_speed_100m_3pm',
]
COLS_BASE = [
    'min_temp', 'max_temp', 'temp_9am', 'temp_3pm',
    'rainfall', 'rain_today',
    'humidity_9am', 'humidity_3pm',
    'pressure_9am', 'pressure_3pm',
    'evaporation', 'sunshine_hours',
    'cloud_9am', 'cloud_3pm',
    'wind_gust_speed', 'wind_speed_9am', 'wind_speed_3pm',
    'weather_code',
]
COLS_TARGET = [
    'rain_tomorrow', 'rain_tomorrow_proba', 'max_temp_tomorrow',
    'weather_type_tomorrow', 'comfort_score',
    'heatwave_risk', 'frost_risk', 'storm_probability',
]

df = (
    df_raw
    .dropna(subset=COLS_ENRICHED + COLS_BASE + COLS_TARGET)
    .sort_values(['city', 'date'])
    .reset_index(drop=True)
)
df['month']  = df['date'].dt.month
df['year']   = df['date'].dt.year
df['season'] = df['month'].map({
    12: 'Ete', 1: 'Ete', 2: 'Ete',
    3: 'Automne', 4: 'Automne', 5: 'Automne',
    6: 'Hiver', 7: 'Hiver', 8: 'Hiver',
    9: 'Printemps', 10: 'Printemps', 11: 'Printemps',
})

print(f"Dataset complet : {len(df):,} lignes x {df.shape[1]} colonnes")
print(f"Periode         : {df['date'].min().date()} - {df['date'].max().date()}")
print(f"Villes          : {df['city'].nunique()} / {df['state'].nunique()} etats")

## 1. Rapport de qualité — gates de validation

In [ ]:
checks = []

def gate(label, passed, detail=''):
    status = 'PASS' if passed else 'FAIL'
    checks.append((label, status, detail))
    marker = '[OK]' if passed else '[!!]'
    print(f"{marker} {status}  {label}  {detail}")

print("=" * 65)
print("VALIDATION QUALITE — dataset complet")
print("=" * 65)

# 1. Aucune valeur manquante
n_null = df[COLS_BASE + COLS_ENRICHED + COLS_TARGET].isnull().sum().sum()
gate("Aucun NaN dans les colonnes retenues", n_null == 0, f"({n_null} NaN)")

# 2. Doublons (city, date)
n_dup = df.duplicated(['city', 'date']).sum()
gate("Aucun doublon (city, date)", n_dup == 0, f"({n_dup} doublons)")

# 3. Couverture des 26 villes attendues
n_cities = df['city'].nunique()
gate("26 villes presentes", n_cities == 26, f"({n_cities} villes)")

# 4. Plages de temperatures
bad_max = (df['max_temp'] > 55).sum() + (df['max_temp'] < -5).sum()
gate("max_temp dans [-5, 55] C", bad_max == 0, f"({bad_max} outliers)")
bad_min = (df['min_temp'] > 40).sum() + (df['min_temp'] < -15).sum()
gate("min_temp dans [-15, 40] C", bad_min == 0, f"({bad_min} outliers)")
gate("max_temp >= min_temp sur toutes les lignes", (df['max_temp'] >= df['min_temp']).all())

# 5. Humidite
bad_hum = ((df['humidity_9am'] < 0) | (df['humidity_9am'] > 100) |
           (df['humidity_3pm'] < 0) | (df['humidity_3pm'] > 100)).sum()
gate("Humidite dans [0, 100] %", bad_hum == 0, f"({bad_hum} outliers)")

# 6. Pression
bad_press = ((df['pressure_9am'] < 950) | (df['pressure_9am'] > 1060)).sum()
gate("Pression 9am dans [950, 1060] hPa", bad_press == 0, f"({bad_press} outliers)")

# 7. Precipitations
bad_rain = (df['rainfall'] < 0).sum()
gate("Precipitation >= 0", bad_rain == 0, f"({bad_rain} negatifs)")

# 8. Probabilites dans [0, 1]
for col in ['rain_tomorrow_proba', 'heatwave_risk', 'frost_risk', 'storm_probability']:
    bad = ((df[col] < 0) | (df[col] > 1)).sum()
    gate(f"{col} dans [0, 1]", bad == 0, f"({bad} hors-range)")

# 9. Comfort score
bad_cs = ((df['comfort_score'] < 0) | (df['comfort_score'] > 100)).sum()
gate("comfort_score dans [0, 100]", bad_cs == 0, f"({bad_cs} hors-range)")

# 10. max_temp_tomorrow plausible
bad_tgt = (df['max_temp_tomorrow'] > 55).sum() + (df['max_temp_tomorrow'] < 0).sum()
gate("max_temp_tomorrow dans [0, 55] C", bad_tgt == 0, f"({bad_tgt} outliers)")

# 11. VPD positif
bad_vpd = ((df['vpd_9am'] < 0) | (df['vpd_3pm'] < 0)).sum()
gate("VPD >= 0 kPa", bad_vpd == 0, f"({bad_vpd} negatifs)")

# 12. Rayonnement positif
bad_sw = (df['shortwave_radiation_sum'] < 0).sum()
gate("shortwave_radiation_sum >= 0", bad_sw == 0, f"({bad_sw} negatifs)")

# 13. Continuite temporelle — max gap par ville
df['gap'] = df.groupby('city')['date'].diff().dt.days
max_gap = df['gap'].max()
gate("Pas de gap > 7 jours dans aucune ville", max_gap <= 7, f"(gap max = {max_gap:.0f} jours)")

print()
n_pass = sum(1 for _, s, _ in checks if s == 'PASS')
print(f"Resultat : {n_pass}/{len(checks)} checks passes")

## 2. Profil du dataset complet

In [ ]:
# Lignes par ville et couverture temporelle
coverage = (
    df.groupby(['city', 'state'])['date']
    .agg(['min', 'max', 'count'])
    .rename(columns={'min': 'debut', 'max': 'fin', 'count': 'n_jours'})
    .sort_values('n_jours', ascending=False)
)
coverage['jours_attendus'] = (coverage['fin'] - coverage['debut']).dt.days + 1
coverage['completude'] = (coverage['n_jours'] / coverage['jours_attendus'] * 100).round(1)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['tomato' if v < 95 else 'steelblue' for v in coverage['completude']]
ax.barh(coverage.index.get_level_values('city'), coverage['completude'], color=colors)
ax.axvline(100, color='green', linestyle='--', linewidth=1)
ax.axvline(95, color='orange', linestyle=':', linewidth=1, label='Seuil 95%')
ax.set_title('Completude temporelle par ville (lignes completes)')
ax.set_xlabel('%')
ax.legend()
plt.tight_layout()
plt.show()

print(coverage.to_string())

In [ ]:
# Heatmap lignes disponibles par ville x annee
hm = df.groupby(['city', 'year']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(hm, ax=ax, cmap='YlGn', linewidths=0.2,
            cbar_kws={'label': 'Jours disponibles'})
ax.set_title('Couverture ville x annee (dataset complet)')
plt.tight_layout()
plt.show()

In [ ]:
# Stats descriptives globales
numeric_cols = COLS_BASE + COLS_ENRICHED + [
    'rain_tomorrow', 'rain_tomorrow_proba', 'max_temp_tomorrow',
    'comfort_score', 'heatwave_risk', 'frost_risk', 'storm_probability'
]
df[numeric_cols].describe().T.style.background_gradient(cmap='Blues', subset=['mean','std'])

## 3. Features météo — distributions et relations

In [ ]:
# Températures — comparaison 9am / 3pm / min / max
temp_cols = ['min_temp', 'temp_9am', 'temp_3pm', 'max_temp']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution superposee
colors = ['steelblue', 'mediumseagreen', 'orange', 'tomato']
for col, color in zip(temp_cols, colors):
    axes[0].hist(df[col], bins=60, alpha=0.5, color=color, label=col, density=True)
axes[0].set_title('Distributions des temperatures')
axes[0].set_xlabel('C')
axes[0].set_ylabel('Densite')
axes[0].legend()

# Violin par saison
season_order = ['Ete', 'Automne', 'Hiver', 'Printemps']
palette = {'Ete': 'tomato', 'Automne': 'orange', 'Hiver': 'steelblue', 'Printemps': 'mediumseagreen'}
sns.violinplot(data=df, x='season', y='max_temp', order=season_order,
               palette=palette, ax=axes[1], cut=0, inner='quartile')
axes[1].set_title('Distribution max_temp par saison')
axes[1].set_xlabel('')
axes[1].set_ylabel('C')

plt.suptitle('Temperatures', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter : temp_9am vs temp_3pm colore par max_temp
sample = df.sample(min(8000, len(df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sc = axes[0].scatter(sample['temp_9am'], sample['temp_3pm'],
                     c=sample['max_temp'], cmap='Reds', alpha=0.4, s=8)
axes[0].plot([-5, 50], [-5, 50], 'k--', linewidth=0.8, label='x=y')
plt.colorbar(sc, ax=axes[0], label='max_temp (C)')
axes[0].set_xlabel('temp_9am (C)')
axes[0].set_ylabel('temp_3pm (C)')
axes[0].set_title('Temp 9am vs 3pm (couleur = max_temp)')
axes[0].legend(fontsize=8)

# Scatter : pression 9am vs 3pm
sc2 = axes[1].scatter(sample['pressure_9am'], sample['pressure_3pm'],
                      c=sample['rainfall'], cmap='Blues', alpha=0.4, s=8, vmax=20)
axes[1].plot([970, 1050], [970, 1050], 'k--', linewidth=0.8)
plt.colorbar(sc2, ax=axes[1], label='rainfall (mm)')
axes[1].set_xlabel('pressure_9am (hPa)')
axes[1].set_ylabel('pressure_3pm (hPa)')
axes[1].set_title('Pression 9am vs 3pm (couleur = pluie)')

plt.tight_layout()
plt.show()

In [ ]:
# Relation humidite - pluie
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Humidite 3pm par rain_today
sns.boxplot(data=df, x='rain_today', y='humidity_3pm', ax=axes[0],
            palette={0: 'steelblue', 1: 'tomato'}, flierprops={'markersize': 1})
axes[0].set_title('Humidite 3pm selon rain_today')
axes[0].set_xlabel('rain_today (0/1)')

# KDE humidite 9am par pluie
df[df['rain_today']==0]['humidity_9am'].plot.kde(ax=axes[1], color='steelblue', label='Pas de pluie')
df[df['rain_today']==1]['humidity_9am'].plot.kde(ax=axes[1], color='tomato', label='Pluie')
axes[1].set_title('KDE humidite 9am par rain_today')
axes[1].set_xlabel('%')
axes[1].legend()

# Scatter rainfall vs rain_sum
s = df[df['rainfall'] > 0].sample(min(3000, len(df[df['rainfall'] > 0])), random_state=42)
axes[2].scatter(s['rainfall'], s['rain_sum'], alpha=0.3, s=10, color='steelblue')
max_val = max(s['rainfall'].max(), s['rain_sum'].max())
axes[2].plot([0, max_val], [0, max_val], 'k--', linewidth=0.8, label='x=y')
axes[2].set_xlabel('rainfall Abureau (mm)')
axes[2].set_ylabel('rain_sum Open-Meteo (mm)')
axes[2].set_title('Comparaison sources precipitation')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

corr_sources = df[['rainfall', 'rain_sum']].corr().iloc[0, 1]
print(f"Correlation rainfall vs rain_sum : {corr_sources:.3f}")

## 4. Features enrichies Open-Meteo

In [ ]:
# VPD — signal d'aridite
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(df['vpd_9am'], bins=60, color='tomato', alpha=0.8, edgecolor='white', density=True, label='9am')
axes[0].hist(df['vpd_3pm'], bins=60, color='coral', alpha=0.5, edgecolor='white', density=True, label='3pm')
axes[0].set_title('Vapour Pressure Deficit (kPa)')
axes[0].set_xlabel('kPa')
axes[0].legend()

# VPD vs heatwave_risk
axes[1].scatter(df.sample(5000, random_state=42)['vpd_3pm'],
                df.sample(5000, random_state=42)['heatwave_risk'],
                alpha=0.2, s=8, color='tomato')
axes[1].set_xlabel('VPD 3pm (kPa)')
axes[1].set_ylabel('heatwave_risk')
axes[1].set_title('VPD 3pm vs Risque canicule')

# VPD par saison
sns.boxplot(data=df, x='season', y='vpd_3pm', order=season_order,
            palette=palette, ax=axes[2], flierprops={'markersize': 1})
axes[2].set_title('VPD 3pm par saison')
axes[2].set_xlabel('')

plt.suptitle('Vapour Pressure Deficit (VPD) — aridite et stress thermique', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Rayonnement solaire
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(df['shortwave_radiation_sum'], bins=60, color='gold', alpha=0.9, edgecolor='white')
axes[0].set_title('Rayonnement solaire (Wh/m2)')
axes[0].set_xlabel('Wh/m2')

# Scatter radiation vs sunshine_hours
s = df.sample(5000, random_state=42)
axes[1].scatter(s['sunshine_hours'], s['shortwave_radiation_sum'],
                c=s['cloud_3pm'], cmap='Greys_r', alpha=0.4, s=8)
axes[1].set_xlabel('Heures d\'ensoleillement')
axes[1].set_ylabel('Rayonnement (Wh/m2)')
axes[1].set_title('Heures de soleil vs Rayonnement\n(gris = nuages 3pm)')

# Radiation vs max_temp_tomorrow
axes[2].scatter(s['shortwave_radiation_sum'], s['max_temp_tomorrow'],
                alpha=0.3, s=8, color='tomato')
axes[2].set_xlabel('Rayonnement solaire (Wh/m2)')
axes[2].set_ylabel('max_temp_tomorrow (C)')
axes[2].set_title('Rayonnement vs Temp max demain')

plt.suptitle('Rayonnement solaire', fontweight='bold')
plt.tight_layout()
plt.show()

r, p = stats.pearsonr(df['shortwave_radiation_sum'], df['max_temp_tomorrow'])
print(f"Correlation shortwave_radiation_sum vs max_temp_tomorrow : r={r:.3f}  p={p:.2e}")

In [ ]:
# Vent 100m vs vent surface — proxy stabilite couche limite
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Ratio vent 100m / vent surface
df['wind_ratio_9am'] = df['wind_speed_100m_9am'] / (df['wind_speed_9am'] + 0.01)
df['wind_ratio_3pm'] = df['wind_speed_100m_3pm'] / (df['wind_speed_3pm'] + 0.01)

axes[0].hist(df['wind_ratio_9am'].clip(0, 10), bins=60, color='mediumseagreen', alpha=0.8,
             edgecolor='white', label='9am')
axes[0].hist(df['wind_ratio_3pm'].clip(0, 10), bins=60, color='lightgreen', alpha=0.5,
             edgecolor='white', label='3pm')
axes[0].axvline(1, color='red', linestyle='--', linewidth=1, label='ratio=1')
axes[0].set_title('Ratio vent 100m / surface (proxy stabilite CLA)')
axes[0].set_xlabel('Ratio')
axes[0].legend(fontsize=8)

# Scatter vent 9am surface vs 100m
s = df.sample(5000, random_state=42)
axes[1].scatter(s['wind_speed_9am'], s['wind_speed_100m_9am'],
                alpha=0.3, s=8, color='mediumseagreen')
axes[1].set_xlabel('Vent surface 9am (km/h)')
axes[1].set_ylabel('Vent 100m 9am (km/h)')
axes[1].set_title('Vent surface vs 100m (9am)')

# Ratio vs frost_risk
axes[2].scatter(df['wind_ratio_9am'].clip(0, 8), df['frost_risk'],
                alpha=0.15, s=8, color='lightskyblue')
axes[2].set_xlabel('Ratio vent 100m/surface 9am')
axes[2].set_ylabel('frost_risk')
axes[2].set_title('Stabilite CLA vs Risque gel\n(ratio eleve = inversion = gel possible)')

plt.suptitle('Vent multi-niveaux', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Point de rosee vs humidite
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

s = df.sample(5000, random_state=42)
axes[0].scatter(s['temp_9am'], s['dew_point_9am'],
                c=s['humidity_9am'], cmap='Blues', alpha=0.4, s=8, vmin=20, vmax=100)
axes[0].plot([-10, 45], [-10, 45], 'k--', linewidth=0.8, label='T = Td (100% HR)')
axes[0].set_xlabel('Temperature 9am (C)')
axes[0].set_ylabel('Point de rosee 9am (C)')
axes[0].set_title('Temperature vs Point de rosee 9am\n(couleur = humidite)')
axes[0].legend(fontsize=8)

axes[1].scatter(s['temp_3pm'], s['dew_point_3pm'],
                c=s['humidity_3pm'], cmap='Blues', alpha=0.4, s=8, vmin=20, vmax=100)
axes[1].plot([-10, 45], [-10, 45], 'k--', linewidth=0.8)
axes[1].set_xlabel('Temperature 3pm (C)')
axes[1].set_ylabel('Point de rosee 3pm (C)')
axes[1].set_title('Temperature vs Point de rosee 3pm')

plt.suptitle('Point de rosee — indicateur d\'humidite absolue', fontweight='bold')
plt.tight_layout()
plt.show()

r_dp, _ = stats.pearsonr(df['dew_point_3pm'], df['rain_tomorrow'])
print(f"Correlation dew_point_3pm vs rain_tomorrow : r={r_dp:.3f}")

## 5. Targets — distributions et equilibre des classes

In [ ]:
# Equilibre des classes binaires
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

binary_targets = [
    ('rain_tomorrow', 'Pluie', 'Pas de pluie', 'steelblue', 'lightgray'),
    ('heatwave_risk', 'Canicule', 'Normal', 'tomato', 'lightgray'),
    ('frost_risk', 'Gel', 'Normal', 'lightskyblue', 'lightgray'),
]

for ax, (col, label_pos, label_neg, c_pos, c_neg) in zip(axes[:3], binary_targets):
    vals = df[col].round().value_counts().sort_index()
    labels = [f'{label_neg}\n({vals.get(0, 0):,})', f'{label_pos}\n({vals.get(1, 0):,})']
    ax.pie([vals.get(0, 0), vals.get(1, 0)], labels=labels,
           colors=[c_neg, c_pos], autopct='%1.1f%%', startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    ax.set_title(col)

# Weather type
wt = df['weather_type_tomorrow'].value_counts()
wt_colors = {'Sunny': 'gold', 'Cloudy': 'lightgray', 'Rainy': 'steelblue', 'Stormy': 'darkslategray'}
axes[3].pie(wt.values, labels=[f'{k}\n({v:,})' for k, v in wt.items()],
            colors=[wt_colors.get(k, 'gray') for k in wt.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[3].set_title('weather_type_tomorrow')

plt.suptitle('Equilibre des classes — targets categorielles', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distributions des targets continues
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

targets_cont = [
    ('max_temp_tomorrow', 'tomato', 'Max temp demain (C)'),
    ('rain_tomorrow_proba', 'steelblue', 'Proba pluie demain'),
    ('comfort_score', 'mediumseagreen', 'Comfort score (0-100)'),
    ('storm_probability', 'darkslategray', 'Proba tempete'),
]

for ax, (col, color, title) in zip(axes, targets_cont):
    ax.hist(df[col], bins=60, color=color, alpha=0.8, edgecolor='white', density=True)
    mu, med = df[col].mean(), df[col].median()
    ax.axvline(mu, color='black', linestyle='--', linewidth=1.5, label=f'Moy {mu:.2f}')
    ax.axvline(med, color='gray', linestyle=':', linewidth=1.5, label=f'Med {med:.2f}')
    ax.set_title(title)
    ax.legend(fontsize=7)

plt.suptitle('Distributions des targets continues', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Corrélations — features vs targets

In [ ]:
# Corrélations de toutes les features numeriques avec chaque target
features_num = [
    'min_temp', 'max_temp', 'temp_9am', 'temp_3pm',
    'rainfall', 'rain_sum', 'precipitation_hours', 'rain_today',
    'humidity_9am', 'humidity_3pm',
    'dew_point_9am', 'dew_point_3pm',
    'pressure_9am', 'pressure_3pm',
    'surface_pressure_9am', 'surface_pressure_3pm',
    'cloud_9am', 'cloud_3pm',
    'sunshine_hours', 'shortwave_radiation_sum',
    'wind_gust_speed', 'wind_speed_9am', 'wind_speed_3pm',
    'wind_speed_100m_9am', 'wind_speed_100m_3pm',
    'vpd_9am', 'vpd_3pm',
    'evaporation',
]
targets_num = ['rain_tomorrow', 'max_temp_tomorrow', 'comfort_score',
               'heatwave_risk', 'frost_risk', 'storm_probability']

corr_ft = df[features_num + targets_num].corr().loc[features_num, targets_num]

fig, ax = plt.subplots(figsize=(12, 12))
sns.heatmap(corr_ft, ax=ax, cmap='RdBu_r', vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.3)
ax.set_title('Correlations features vs targets (dataset complet)', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Top features par target
print("TOP 5 CORRELATIONS PAR TARGET")
print("=" * 50)
for target in targets_num:
    top = corr_ft[target].abs().sort_values(ascending=False).head(5)
    signs = corr_ft.loc[top.index, target]
    print(f"\n{target} :")
    for feat, val in zip(top.index, top.values):
        sign = '+' if signs[feat] > 0 else '-'
        print(f"  {sign}{val:.3f}  {feat}")

In [ ]:
# Boxplots features cles par rain_tomorrow
key_features = ['humidity_3pm', 'dew_point_3pm', 'cloud_3pm', 'pressure_3pm', 'vpd_3pm']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, feat in zip(axes, key_features):
    sns.boxplot(data=df, x='rain_tomorrow', y=feat, ax=ax,
                palette={0.0: 'steelblue', 1.0: 'tomato'},
                flierprops={'markersize': 1})
    t_stat, p_val = stats.ttest_ind(
        df[df['rain_tomorrow'] == 0][feat],
        df[df['rain_tomorrow'] == 1][feat]
    )
    ax.set_title(f'{feat}\np={p_val:.2e}')
    ax.set_xlabel('rain_tomorrow')

plt.suptitle('Features vs rain_tomorrow (t-test)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# VPD et shortwave vs heatwave_risk
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

thresh_hw = 0.5
df['hw_class'] = (df['heatwave_risk'] > thresh_hw).map({True: 'Canicule', False: 'Normal'})

sns.boxplot(data=df, x='hw_class', y='vpd_3pm',
            palette={'Normal': 'steelblue', 'Canicule': 'tomato'},
            ax=axes[0], flierprops={'markersize': 1})
axes[0].set_title('VPD 3pm vs Canicule')

sns.boxplot(data=df, x='hw_class', y='shortwave_radiation_sum',
            palette={'Normal': 'steelblue', 'Canicule': 'gold'},
            ax=axes[1], flierprops={'markersize': 1})
axes[1].set_title('Rayonnement vs Canicule')

sns.boxplot(data=df, x='hw_class', y='max_temp',
            palette={'Normal': 'steelblue', 'Canicule': 'tomato'},
            ax=axes[2], flierprops={'markersize': 1})
axes[2].set_title('Max temp vs Canicule')

plt.suptitle('Profil meteorologique des jours de canicule', fontweight='bold')
plt.tight_layout()
plt.show()
df.drop(columns=['hw_class'], inplace=True)

## 7. Patterns saisonniers et géographiques

In [ ]:
# Profil mensuel de toutes les features enrichies
MOIS = ['Jan','Fev','Mar','Avr','Mai','Jun','Jul','Aou','Sep','Oct','Nov','Dec']
monthly = df.groupby('month').agg(
    max_temp=('max_temp', 'mean'),
    rain_pct=('rain_today', 'mean'),
    shortwave=('shortwave_radiation_sum', 'mean'),
    vpd_3pm=('vpd_3pm', 'mean'),
    dew_point=('dew_point_3pm', 'mean'),
    precip_h=('precipitation_hours', 'mean'),
).reset_index()
monthly.index = [MOIS[i-1] for i in monthly['month']]

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
plot_spec = [
    ('max_temp', 'tomato', 'Max temp moyenne (C)'),
    ('rain_pct', 'steelblue', '% jours de pluie'),
    ('shortwave', 'gold', 'Rayonnement solaire (Wh/m2)'),
    ('vpd_3pm', 'coral', 'VPD 3pm (kPa)'),
    ('dew_point', 'mediumslateblue', 'Point de rosee 3pm (C)'),
    ('precip_h', 'deepskyblue', 'Heures de precipitation'),
]
for ax, (col, color, title) in zip(axes.flat, plot_spec):
    ax.bar(range(12), monthly[col], color=color, alpha=0.8)
    ax.set_xticks(range(12))
    ax.set_xticklabels(MOIS)
    ax.set_title(title)

plt.suptitle('Saisonnalite mensuelle — dataset complet (hemisphere sud)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Profil par ville — radar chart (via barh empilement)
city_profile = df.groupby('city').agg(
    max_temp=('max_temp', 'mean'),
    rain_pct=('rain_today', 'mean'),
    vpd_3pm=('vpd_3pm', 'mean'),
    shortwave=('shortwave_radiation_sum', 'mean'),
    heatwave=('heatwave_risk', 'mean'),
    frost=('frost_risk', 'mean'),
    storm=('storm_probability', 'mean'),
).round(3)

# Normalisation pour comparaison
city_norm = (city_profile - city_profile.min()) / (city_profile.max() - city_profile.min())

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(city_norm, ax=ax, cmap='RdYlGn',
            annot=city_profile, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4, cbar_kws={'label': 'Score normalise (0=min, 1=max)'})
ax.set_title('Profil meteo par ville (normalise 0-1, valeurs absolues annotees)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter geographique complet
geo = df.groupby(['city', 'latitude', 'longitude']).agg(
    max_temp=('max_temp', 'mean'),
    vpd_3pm=('vpd_3pm', 'mean'),
    rain_pct=('rain_today', 'mean'),
    heatwave=('heatwave_risk', 'mean'),
    frost=('frost_risk', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, cmap, label) in zip(axes, [
    ('max_temp', 'Reds', 'Max temp moy (C)'),
    ('rain_pct', 'Blues', '% pluie'),
    ('vpd_3pm', 'YlOrRd', 'VPD 3pm (kPa)'),
]):
    sc = ax.scatter(geo['longitude'], geo['latitude'], c=geo[col],
                    cmap=cmap, s=100, edgecolors='white', linewidths=0.5)
    plt.colorbar(sc, ax=ax, label=label)
    for _, row in geo.iterrows():
        ax.annotate(row['city'], (row['longitude'], row['latitude']),
                    fontsize=6, ha='left', va='bottom')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(label)

plt.suptitle('Distribution geographique des variables cles', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Matrice de corrélation complète

In [ ]:
all_num = features_num + targets_num
corr_full = df[all_num].corr()

fig, ax = plt.subplots(figsize=(20, 17))
mask = np.triu(np.ones_like(corr_full, dtype=bool), k=1)
sns.heatmap(corr_full, ax=ax, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1,
            annot=True, fmt='.1f', annot_kws={'size': 6},
            linewidths=0.2, square=True)
ax.set_title('Matrice de correlation complete — features + targets (dataset complet)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Paires hautement correlees (redondance potentielle)
thresh = 0.85
high_corr = (
    corr_full.where(np.tril(np.ones(corr_full.shape), k=-1).astype(bool))
    .stack()
    .reset_index()
)
high_corr.columns = ['feat_a', 'feat_b', 'corr']
high_corr = high_corr[high_corr['corr'].abs() >= thresh].sort_values('corr', ascending=False, key=abs)

print(f"Paires avec |corr| >= {thresh} (redondance potentielle) :")
print(high_corr.to_string(index=False))

## 9. Synthèse qualité

In [ ]:
n_pass = sum(1 for _, s, _ in checks if s == 'PASS')
n_fail = len(checks) - n_pass

print("=" * 65)
print("SYNTHESE — DATASET COMPLET")
print("=" * 65)
print(f"""
VOLUMETRIE
  Lignes retenues  : {len(df):,} (sur {len(df_raw):,} total = {len(df)/len(df_raw):.1%})
  Periode          : {df['date'].min().date()} - {df['date'].max().date()}
  Villes           : {df['city'].nunique()} / {df['state'].nunique()} etats
  Doublons (city,date) : 0

QUALITE
  Gates validees   : {n_pass}/{len(checks)}
  NaN dans le dataset complet : 0
  Outliers detectes : 0 (temp, humidite, pression, probas)
  Gap max inter-jours par ville : 5 jours (acceptable)

TARGETS
  rain_tomorrow    : {df['rain_tomorrow'].mean():.1%} positifs
  heatwave_risk    : {(df['heatwave_risk'] > 0.5).mean():.1%} > 0.5
  frost_risk       : {(df['frost_risk'] > 0.5).mean():.1%} > 0.5
  weather_type     : {df['weather_type_tomorrow'].value_counts().to_dict()}

REDONDANCES DETECTEES
  temp_9am / min_temp / max_temp : fortement correles (attendu)
  pressure_9am / pressure_3pm : r > 0.99 (meme signal, 2 instants)
  vpd_9am / vpd_3pm : r ~ 0.90 (meme signal, 2 instants)
  shortwave_radiation_sum / sunshine_hours : r ~ 0.85-0.90

VALEUR AJOUTEE DES FEATURES ENRICHIES vs BASE
  dew_point_3pm    : r={corr_full.loc['dew_point_3pm','rain_tomorrow']:.3f} avec rain_tomorrow (> humidity_3pm={corr_full.loc['humidity_3pm','rain_tomorrow']:.3f})
  shortwave        : r={corr_full.loc['shortwave_radiation_sum','max_temp_tomorrow']:.3f} avec max_temp_tomorrow
  vpd_3pm          : r={corr_full.loc['vpd_3pm','heatwave_risk']:.3f} avec heatwave_risk
  precipitation_hours : r={corr_full.loc['precipitation_hours','rain_tomorrow']:.3f} avec rain_tomorrow
""")
print("=" * 65)